# Transformer KV Cache 推理优化教程

本 notebook 通过 PyTorch 从零实现 **KV Cache**，深入讲解其原理、实现与性能收益。

**KV Cache** 是现代大语言模型（GPT、LLaMA、ChatGPT 等）推理加速的核心技术。  
没有 KV Cache，自回归生成的计算量随生成长度**平方增长**；  
有了 KV Cache，每步只需计算新 token，复杂度降至**线性**。

## 目录
1. 为什么需要 KV Cache？（问题背景 + 计算量可视化）
2. Decoder-Only Transformer 基础实现（不带 KV Cache）
3. KV Cache 原理与实现
4. 正确性验证（两版实现输出完全一致）
5. 速度对比实验
6. 内存占用分析
7. 注意力模式可视化（Prefill vs Decode）
8. 总结

## 环境与依赖

In [ ]:
import math
import time
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from typing import Optional, Tuple, List

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用设备: {DEVICE}')
print(f'PyTorch 版本: {torch.__version__}')

## 1. 为什么需要 KV Cache？

### 自回归生成过程

Transformer 的生成过程是**自回归**的：每次只生成一个 token，新 token 基于所有已生成 token 的上下文。

### 朴素实现的瓶颈

不带 KV Cache 时，生成第 $t$ 个 token 的流程：
1. 将整个已有序列（长度 $t$）输入模型
2. 对**所有 $t$ 个 token** 重新计算 Key、Value 投影
3. 运行完整注意力计算

| 生成步骤 | 序列长度 | K/V 投影计算量 |
|---------|---------|---------------|
| 步骤 1  | 1       | 1 个 token    |
| 步骤 2  | 2       | 2 个 token    |
| 步骤 T  | T       | T 个 token    |
| **合计** | —      | $\frac{T(T+1)}{2}$ → $O(T^2)$ |

### KV Cache 的解决思路

**关键观察**：在因果（Causal）注意力中，位置 $i$ 的 Key、Value 只由位置 $i$ 的输入决定，**不会随后续 token 的加入而改变**。

因此可以**缓存**已计算的 K、V，下一步直接复用：
- **Prefill 阶段**：一次性处理完整 prompt，保存所有 K、V
- **Decode 阶段**：每步只计算新 token 的 Q、K、V，拼接到缓存后做注意力

| 阶段 | 计算量 |
|------|-------|
| Prefill（长度 P） | $P$ |
| Decode（生成 G 个 token） | $G \times 1$ |
| **合计** | $P + G$ → $O(P + G)$ |

In [ ]:
# ─── 计算量对比可视化 ─────────────────────────────────────
gen_lengths = list(range(1, 101))
P = 16  # prompt 长度固定

# 无 KV Cache：sum_{t=1}^{P+G} t (朴素自回归，每步处理完整序列)
ops_no_cache = [sum(range(1, P + G + 1)) for G in gen_lengths]

# 有 KV Cache：P (prefill) + G * 1 (decode)
ops_with_cache = [P + G for G in gen_lengths]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图：绝对计算量
ax = axes[0]
ax.plot(gen_lengths, ops_no_cache,   'r-o', markersize=4, linewidth=2, label='No KV Cache  O(T²)')
ax.plot(gen_lengths, ops_with_cache, 'b-s', markersize=4, linewidth=2, label='With KV Cache O(T)')
ax.set_xlabel('Generated Tokens (G)', fontsize=12)
ax.set_ylabel('Cumulative K/V Projection Ops (relative)', fontsize=11)
ax.set_title(f'Total K/V Computation  (prompt length P={P})', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# 右图：加速比
ax2 = axes[1]
speedup = [nc / wc for nc, wc in zip(ops_no_cache, ops_with_cache)]
ax2.plot(gen_lengths, speedup, 'g-^', markersize=4, linewidth=2)
ax2.axhline(y=1, color='gray', linestyle='--', linewidth=1)
ax2.set_xlabel('Generated Tokens (G)', fontsize=12)
ax2.set_ylabel('Speedup (theoretical)', fontsize=12)
ax2.set_title('Theoretical Speedup of KV Cache', fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.fill_between(gen_lengths, 1, speedup, alpha=0.15, color='green')

plt.suptitle('KV Cache: Computational Cost Reduction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('kv_cache_complexity.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'生成 100 个 token 时，KV Cache 理论加速比: {speedup[-1]:.1f}x')

## 2. Decoder-Only Transformer 基础实现（不带 KV Cache）

KV Cache 主要用于 **Decoder-Only** 架构（GPT 系列、LLaMA 等）。

与 Encoder-Decoder 不同，Decoder-Only 模型：
- 只有 Decoder 部分（无 Encoder）
- 使用**因果（Causal）自注意力**：每个位置只能看到左边的 token
- 训练和推理时均使用下三角掩码

下面先实现不带 KV Cache 的版本，理解推理瓶颈所在。

In [ ]:
class CausalSelfAttention(nn.Module):
    """因果自注意力（不带 KV Cache）"""

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, 'd_model 必须能被 num_heads 整除'
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.last_attn_weights = None  # 保存供可视化

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, T, d_model)  — 完整序列
        对所有 T 个 token 重新计算 Q, K, V（推理时的瓶颈）
        """
        B, T, _ = x.shape

        # 对所有 T 个位置计算 Q, K, V
        Q = self.W_q(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        # 每次都重新计算所有 T 个 token 的 K, V ← 推理瓶颈

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # 因果掩码：下三角矩阵，位置 i 只能看到 0..i
        causal_mask = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool))
        scores = scores.masked_fill(~causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))

        attn = F.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        self.last_attn_weights = attn.detach()

        out = torch.matmul(self.dropout(attn), V)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_o(out)


# 快速验证
attn_no_cache = CausalSelfAttention(d_model=64, num_heads=4)
x = torch.randn(2, 10, 64)
out = attn_no_cache(x)
print('CausalSelfAttention 输出 shape:', out.shape)  # (2, 10, 64)

In [ ]:
class GPTBlock(nn.Module):
    """GPT Decoder Block（不带 KV Cache）"""

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.attn = CausalSelfAttention(d_model, num_heads, dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x))   # Pre-LN + 残差
        x = x + self.ff(self.norm2(x))
        return x


class GPTDecoder(nn.Module):
    """GPT-style Decoder（不带 KV Cache）"""

    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        num_heads: int,
        num_layers: int,
        d_ff: int,
        max_len: int,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.max_len = max_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb   = nn.Embedding(max_len, d_model)   # 可学习位置编码
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            GPTBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) token indices"""
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)   # (1, T)
        x = self.drop(self.token_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.norm(x))   # (B, T, vocab_size)

    @torch.no_grad()
    def generate_no_cache(self, idx: torch.Tensor, max_new_tokens: int) -> torch.Tensor:
        """自回归生成 —— 朴素实现（每步处理完整序列）"""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_len:]        # 截断到 max_len
            logits = self(idx_cond)                  # 整个序列重新前向
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            idx = torch.cat([idx, next_token], dim=1)
        return idx


# 验证
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

model_no_cache = GPTDecoder(
    vocab_size=500, d_model=128, num_heads=4,
    num_layers=4, d_ff=512, max_len=256
).to(DEVICE)

print(f'GPTDecoder 参数量: {count_params(model_no_cache):,}')
sample_ids = torch.randint(1, 500, (1, 8)).to(DEVICE)
logits = model_no_cache(sample_ids)
print(f'前向传播输出 shape: {logits.shape}')   # (1, 8, 500)

## 3. KV Cache 原理与实现

### 核心数学原理

在因果注意力中，位置 $i$ 的 Key 和 Value 投影为：

$$K_i = W_K \cdot \text{LayerNorm}(x_i), \quad V_i = W_V \cdot \text{LayerNorm}(x_i)$$

其中 $x_i$ 是第 $i$ 个位置在**当前层的输入**。由于因果约束，$x_i$ 只依赖于位置 $0, 1, \ldots, i$，与之后的位置无关。

因此，**已经计算过的 $K_i, V_i$ 在后续步骤中完全不变**，可以安全缓存。

### 两阶段推理

```
┌─────────────────────────────────────────────────────────────┐
│  Prefill 阶段（处理 Prompt）                                  │
│                                                              │
│  输入: [t₀, t₁, t₂, ..., tₚ]                               │
│  操作: 完整前向传播，保存每层的 K, V → KV Cache              │
│  输出: 第一个新 token                                         │
└─────────────────────────────────────────────────────────────┘
              ↓
┌─────────────────────────────────────────────────────────────┐
│  Decode 阶段（逐步生成）                                      │
│                                                              │
│  输入: [新 token]（只有 1 个！）                              │
│  操作: 计算新 token 的 Q, K, V                               │
│        从 Cache 读取历史 K, V                                 │
│        K_full = [K_cache; K_new]                             │
│        V_full = [V_cache; V_new]                             │
│        注意力: Q_new × K_full^T × V_full                     │
│        更新 Cache: 追加新的 K, V                              │
│  输出: 下一个 token                                           │
└─────────────────────────────────────────────────────────────┘
```

### 实现要点

1. **位置编码**：Decode 阶段输入只有 1 个 token，但其位置是 `past_len`（历史长度），需要正确传入
2. **因果掩码**：Decode 阶段 Query 是最新位置，所有 Cache 中的 Key 均在过去，不需要额外掩码
3. **Prefill 阶段**：序列长度 > 1 时仍需正常的下三角因果掩码
4. **每层独立缓存**：KV Cache 是 **per-layer** 的，每个 Transformer 层维护自己的缓存

In [ ]:
class CausalSelfAttentionKV(nn.Module):
    """因果自注意力（带 KV Cache 支持）"""

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.last_attn_weights = None

    def forward(
        self,
        x: torch.Tensor,
        past_kv: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]]]:
        """
        x       : (B, T, d_model)  T=序列长度（Decode 阶段 T=1）
        past_kv : (K_cache, V_cache)，各自形状 (B, heads, past_len, d_k)
        use_cache: True 时返回更新后的 KV Cache

        Returns:
            out    : (B, T, d_model)
            new_kv : 更新后的 (K, V)，or None
        """
        B, T, _ = x.shape

        # ── 只计算新 token 的 Q, K, V ──────────────────────────
        Q = self.W_q(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        # Decode 阶段：T=1，只做 1 个 token 的投影 ← 核心节省

        # ── 拼接历史 KV Cache ────────────────────────────────
        if past_kv is not None:
            K_cache, V_cache = past_kv
            K = torch.cat([K_cache, K], dim=2)   # (B, heads, past_len+T, d_k)
            V = torch.cat([V_cache, V], dim=2)

        total_len = K.size(2)    # past_len + T
        past_len  = total_len - T

        # ── 计算注意力分数 ───────────────────────────────────
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        # (B, heads, T, total_len)

        # ── 因果掩码 ─────────────────────────────────────────
        # Query[i]（绝对位置 past_len+i）只能看到 Key[j]，j <= past_len+i
        # Decode 阶段 T=1：只有 1 个 Query，所有 Cache 均在过去，无需掩码
        if T > 1:
            row = torch.arange(T, device=x.device).view(T, 1)          # (T, 1)
            col = torch.arange(total_len, device=x.device).view(1, total_len)  # (1, total_len)
            causal_mask = col <= (past_len + row)                       # (T, total_len)
            scores = scores.masked_fill(
                ~causal_mask.unsqueeze(0).unsqueeze(0), float('-inf')
            )

        attn = F.softmax(scores, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        self.last_attn_weights = attn.detach()

        out = torch.matmul(self.dropout(attn), V)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)

        # ── 更新并返回 Cache ─────────────────────────────────
        new_kv = (K.detach(), V.detach()) if use_cache else None
        return self.W_o(out), new_kv


# 验证：Decode 阶段（T=1）with Cache
attn_kv = CausalSelfAttentionKV(d_model=64, num_heads=4)
x_new = torch.randn(2, 1, 64)   # 单个新 token
K_cache = torch.randn(2, 4, 5, 16)  # 5 个历史位置的 K
V_cache = torch.randn(2, 4, 5, 16)  # 5 个历史位置的 V
out, new_kv = attn_kv(x_new, past_kv=(K_cache, V_cache), use_cache=True)
print('Decode 阶段输出 shape:', out.shape)       # (2, 1, 64)
print('更新后 K cache shape:', new_kv[0].shape)  # (2, 4, 6, 16) ← 多了 1 步

In [ ]:
class GPTBlockKV(nn.Module):
    """GPT Block（带 KV Cache 支持）"""

    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.attn = CausalSelfAttentionKV(d_model, num_heads, dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(
        self,
        x: torch.Tensor,
        past_kv: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Tuple[torch.Tensor, torch.Tensor]]]:
        attn_out, new_kv = self.attn(self.norm1(x), past_kv, use_cache)
        x = x + attn_out
        x = x + self.ff(self.norm2(x))
        return x, new_kv


class GPTDecoderKV(nn.Module):
    """GPT-style Decoder（带 KV Cache 支持）"""

    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        num_heads: int,
        num_layers: int,
        d_ff: int,
        max_len: int,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.max_len = max_len
        self.num_layers = num_layers
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb   = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            GPTBlockKV(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(
        self,
        idx: torch.Tensor,
        past_kv_list: Optional[List] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[List]]:
        """
        idx          : (B, T)  T=1 in decode phase
        past_kv_list : list of per-layer (K_cache, V_cache)
        use_cache    : True 时返回更新后的 KV Cache list
        """
        B, T = idx.shape

        # 计算当前 token 的绝对位置
        past_len = 0
        if past_kv_list is not None and past_kv_list[0] is not None:
            past_len = past_kv_list[0][0].size(2)   # K_cache.shape[2]

        # 位置编码：past_len, past_len+1, ..., past_len+T-1
        pos = torch.arange(past_len, past_len + T, device=idx.device).unsqueeze(0)
        x = self.drop(self.token_emb(idx) + self.pos_emb(pos))

        new_kv_list = []
        for i, block in enumerate(self.blocks):
            past_kv = past_kv_list[i] if past_kv_list is not None else None
            x, new_kv = block(x, past_kv=past_kv, use_cache=use_cache)
            new_kv_list.append(new_kv)

        logits = self.lm_head(self.norm(x))   # (B, T, vocab_size)
        return logits, (new_kv_list if use_cache else None)

    @torch.no_grad()
    def generate_with_cache(self, idx: torch.Tensor, max_new_tokens: int) -> torch.Tensor:
        """自回归生成 —— 带 KV Cache（每步只处理 1 个新 token）"""
        # ── Prefill 阶段：处理整个 prompt ────────────────────
        logits, kv_cache = self(idx, use_cache=True)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        idx = torch.cat([idx, next_token], dim=1)

        # ── Decode 阶段：逐步生成，每步只处理 1 个 token ─────
        for _ in range(max_new_tokens - 1):
            logits, kv_cache = self(next_token, past_kv_list=kv_cache, use_cache=True)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            idx = torch.cat([idx, next_token], dim=1)

        return idx


model_kv = GPTDecoderKV(
    vocab_size=500, d_model=128, num_heads=4,
    num_layers=4, d_ff=512, max_len=256
).to(DEVICE)

print(f'GPTDecoderKV 参数量: {count_params(model_kv):,}')
logits_kv, cache = model_kv(sample_ids, use_cache=True)
print(f'前向输出 shape: {logits_kv.shape}')           # (1, 8, 500)
print(f'KV Cache 层数: {len(cache)}')
print(f'Layer 0 K cache shape: {cache[0][0].shape}')  # (1, 4, 8, 32)

## 4. 正确性验证

带 KV Cache 的版本应当与朴素版本产生**完全一致的输出**。

**数学保证**：因为 Pre-LN 结构下，层 $l$ 第 $i$ 个位置的隐状态 $h_l^{(i)}$ 只依赖于位置 $0, \ldots, i$ 的输入（因果性），所以：
- Prefill 阶段计算出的 $K_l^{(i)}, V_l^{(i)}$ 等于全序列计算时的值
- Decode 阶段新 token 的 $K, V$ 也与全序列计算时的值一致

下面用相同权重验证两个版本输出完全匹配。

In [ ]:
# ─── 创建两个模型，共享相同权重 ──────────────────────────────
VOCAB  = 200
D_MODEL = 64
HEADS   = 4
LAYERS  = 2
D_FF    = 256
MAX_LEN = 128

# 朴素版本
model_ref = GPTDecoder(
    vocab_size=VOCAB, d_model=D_MODEL, num_heads=HEADS,
    num_layers=LAYERS, d_ff=D_FF, max_len=MAX_LEN
).to(DEVICE).eval()

# KV Cache 版本
model_cache = GPTDecoderKV(
    vocab_size=VOCAB, d_model=D_MODEL, num_heads=HEADS,
    num_layers=LAYERS, d_ff=D_FF, max_len=MAX_LEN
).to(DEVICE).eval()

# 两个模型的参数名结构相同，直接用 state_dict 拷贝权重
model_cache.load_state_dict(model_ref.state_dict())
print('权重已对齐 ✓')

# ─── 生成测试 ────────────────────────────────────────────
torch.manual_seed(0)
prompt = torch.randint(1, VOCAB, (1, 12)).to(DEVICE)
N_NEW = 8

with torch.no_grad():
    out_ref   = model_ref.generate_no_cache(prompt.clone(), N_NEW)
    out_cache = model_cache.generate_with_cache(prompt.clone(), N_NEW)

print(f'\nPrompt  : {prompt[0].tolist()}')
print(f'No Cache: {out_ref[0].tolist()}')
print(f'KV Cache: {out_cache[0].tolist()}')

match = torch.all(out_ref == out_cache).item()
print(f'\n两版本输出完全一致: {match} ✓' if match else '\n❌ 输出不一致，请检查实现！')

## 5. 速度对比实验

对比不同生成长度下，两个版本的推理时间。

In [ ]:
# ─── 基准测试配置 ─────────────────────────────────────────
BENCH_VOCAB    = 1000
BENCH_D_MODEL  = 256
BENCH_HEADS    = 8
BENCH_LAYERS   = 6
BENCH_D_FF     = 1024
BENCH_MAX_LEN  = 256
PROMPT_LEN     = 16
GEN_LENGTHS    = [8, 16, 32, 64, 96]
N_RUNS         = 3

bench_ref = GPTDecoder(
    vocab_size=BENCH_VOCAB, d_model=BENCH_D_MODEL, num_heads=BENCH_HEADS,
    num_layers=BENCH_LAYERS, d_ff=BENCH_D_FF, max_len=BENCH_MAX_LEN
).to(DEVICE).eval()

bench_kv = GPTDecoderKV(
    vocab_size=BENCH_VOCAB, d_model=BENCH_D_MODEL, num_heads=BENCH_HEADS,
    num_layers=BENCH_LAYERS, d_ff=BENCH_D_FF, max_len=BENCH_MAX_LEN
).to(DEVICE).eval()
bench_kv.load_state_dict(bench_ref.state_dict())

print(f'Benchmark 模型参数量: {count_params(bench_ref):,}')


def measure_time(fn, n_runs=N_RUNS):
    with torch.no_grad():
        for _ in range(1):  # warmup
            fn()
        times = []
        for _ in range(n_runs):
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            fn()
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    return np.mean(times) * 1000  # ms


results = {'gen_len': [], 'no_cache_ms': [], 'kv_cache_ms': [], 'speedup': []}
prompt_bench = torch.randint(1, BENCH_VOCAB, (1, PROMPT_LEN)).to(DEVICE)

print(f'\n{"Gen Len":>8} {"No Cache (ms)":>15} {"KV Cache (ms)":>15} {"Speedup":>9}')
print('-' * 52)

for gen_len in GEN_LENGTHS:
    t_no = measure_time(lambda: bench_ref.generate_no_cache(prompt_bench.clone(), gen_len))
    t_kv = measure_time(lambda: bench_kv.generate_with_cache(prompt_bench.clone(), gen_len))
    sp   = t_no / t_kv
    results['gen_len'].append(gen_len)
    results['no_cache_ms'].append(t_no)
    results['kv_cache_ms'].append(t_kv)
    results['speedup'].append(sp)
    print(f'{gen_len:>8} {t_no:>15.1f} {t_kv:>15.1f} {sp:>8.2f}x')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
x_pos = np.arange(len(GEN_LENGTHS))
width = 0.35
bars1 = ax.bar(x_pos - width/2, results['no_cache_ms'], width, label='No KV Cache',   color='salmon',     edgecolor='black', linewidth=0.5)
bars2 = ax.bar(x_pos + width/2, results['kv_cache_ms'], width, label='With KV Cache', color='steelblue',  edgecolor='black', linewidth=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels([str(g) for g in GEN_LENGTHS])
ax.set_xlabel('Generated Tokens', fontsize=12)
ax.set_ylabel('Time (ms)', fontsize=12)
ax.set_title('Generation Time: No Cache vs KV Cache', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, axis='y', alpha=0.3)

ax2 = axes[1]
ax2.plot(GEN_LENGTHS, results['speedup'], 'g-o', markersize=8, linewidth=2.5)
ax2.fill_between(GEN_LENGTHS, 1, results['speedup'], alpha=0.15, color='green')
ax2.axhline(y=1, color='gray', linestyle='--', linewidth=1)
ax2.set_xlabel('Generated Tokens', fontsize=12)
ax2.set_ylabel('Speedup (×)', fontsize=12)
ax2.set_title('KV Cache Speedup', fontsize=12)
ax2.grid(True, alpha=0.3)
for x, y in zip(GEN_LENGTHS, results['speedup']):
    ax2.annotate(f'{y:.1f}×', (x, y), textcoords='offset points', xytext=(0, 8),
                 ha='center', fontsize=10, fontweight='bold')

plt.suptitle(f'Benchmark: prompt={PROMPT_LEN}, d_model={BENCH_D_MODEL}, layers={BENCH_LAYERS}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('kv_cache_benchmark.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. 内存占用分析

KV Cache 以**空间换时间**，需要分析其内存开销。

### 内存公式

$$\text{KV Cache 内存} = L \times 2 \times B \times H \times S \times d_k \times \text{bytes}$$

其中：
- $L$ = Transformer 层数
- $2$ = K 和 V
- $B$ = batch size
- $H$ = 注意力头数
- $S$ = 序列长度
- $d_k = d_{\text{model}} / H$
- bytes = 每个元素字节数（float32=4, float16=2）

化简：$= L \times 2 \times B \times d_{\text{model}} \times S \times \text{bytes}$

In [ ]:
def kv_cache_memory_mb(num_layers, d_model, seq_len, batch_size=1, dtype_bytes=2):
    """计算 KV Cache 占用内存（MB）"""
    return (num_layers * 2 * batch_size * d_model * seq_len * dtype_bytes) / (1024 ** 2)


# ── 不同序列长度下的内存占用 ─────────────────────────────────
seq_lengths_mem = [128, 256, 512, 1024, 2048, 4096, 8192]

configs = {
    'Small (d=512, L=6)':    {'num_layers': 6,  'd_model': 512},
    'Medium (d=1024, L=12)': {'num_layers': 12, 'd_model': 1024},
    'Large (d=4096, L=32)':  {'num_layers': 32, 'd_model': 4096},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for label, cfg in configs.items():
    mems = [kv_cache_memory_mb(cfg['num_layers'], cfg['d_model'], s, dtype_bytes=2)
            for s in seq_lengths_mem]
    ax.plot(seq_lengths_mem, mems, '-o', markersize=5, linewidth=2, label=label)
ax.set_xlabel('Sequence Length', fontsize=12)
ax.set_ylabel('KV Cache Memory (MB, float16, batch=1)', fontsize=10)
ax.set_title('KV Cache Memory vs Sequence Length', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xscale('log', base=2)
ax.set_yscale('log', base=10)

# ── 真实 LLM 对比 ─────────────────────────────────────────────
ax2 = axes[1]
real_models = {
    'GPT-2 Small\n(L=12, d=768)':   {'num_layers': 12, 'd_model': 768},
    'GPT-2 XL\n(L=48, d=1600)':     {'num_layers': 48, 'd_model': 1600},
    'LLaMA-7B\n(L=32, d=4096)':     {'num_layers': 32, 'd_model': 4096},
    'LLaMA-70B\n(L=80, d=8192)':    {'num_layers': 80, 'd_model': 8192},
}
bar_labels = list(real_models.keys())
bar_mems_2k  = [kv_cache_memory_mb(c['num_layers'], c['d_model'], 2048, dtype_bytes=2) for c in real_models.values()]
bar_mems_8k  = [kv_cache_memory_mb(c['num_layers'], c['d_model'], 8192, dtype_bytes=2) for c in real_models.values()]

x_b = np.arange(len(bar_labels))
ax2.bar(x_b - 0.2, bar_mems_2k, 0.38, label='seq=2048', color='steelblue',  edgecolor='k', linewidth=0.5)
ax2.bar(x_b + 0.2, bar_mems_8k, 0.38, label='seq=8192', color='darkorange', edgecolor='k', linewidth=0.5)
ax2.set_xticks(x_b)
ax2.set_xticklabels(bar_labels, fontsize=9)
ax2.set_ylabel('KV Cache Memory (MB, float16, batch=1)', fontsize=10)
ax2.set_title('KV Cache for Real LLMs', fontsize=12)
ax2.legend(fontsize=10)
ax2.set_yscale('log', base=10)
ax2.grid(True, axis='y', alpha=0.3)

plt.suptitle('KV Cache Memory Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('kv_cache_memory.png', dpi=120, bbox_inches='tight')
plt.show()

# ── 打印具体数值 ────────────────────────────────────────────
print('\n真实 LLM KV Cache 内存（float16, batch=1）:')
print(f'{"Model":>30} {"seq=2048 (MB)":>15} {"seq=8192 (MB)":>15}')
print('-' * 63)
for label, mem2k, mem8k in zip(bar_labels, bar_mems_2k, bar_mems_8k):
    label_clean = label.replace('\n', ' ')
    print(f'{label_clean:>30} {mem2k:>15.1f} {mem8k:>15.1f}')

## 7. 注意力模式可视化

通过可视化注意力分数矩阵，直观理解 Prefill 和 Decode 阶段的区别：

- **Prefill 阶段**：所有位置同时计算，产生标准下三角注意力矩阵
- **Decode 阶段**：只有 1 个 Query（新 token），对所有历史 Key 进行注意力

In [ ]:
torch.manual_seed(42)

VIS_VOCAB = 50
VIS_D = 32
VIS_HEADS = 2

vis_model = GPTDecoderKV(
    vocab_size=VIS_VOCAB, d_model=VIS_D, num_heads=VIS_HEADS,
    num_layers=1, d_ff=128, max_len=64, dropout=0.0
).to(DEVICE).eval()

PROMPT_LEN_VIS = 6
prompt_vis = torch.randint(1, VIS_VOCAB, (1, PROMPT_LEN_VIS)).to(DEVICE)

# ── 提取各阶段注意力权重 ──────────────────────────────────────
with torch.no_grad():
    # Prefill 阶段
    logits_pf, kv_cache_vis = vis_model(prompt_vis, use_cache=True)
    attn_prefill = vis_model.blocks[0].attn.last_attn_weights[0]  # (heads, P, P)

    decode_attn_steps = []
    curr_token = logits_pf[:, -1, :].argmax(-1, keepdim=True)
    for step in range(4):   # 记录 4 个 decode 步
        logits_d, kv_cache_vis = vis_model(
            curr_token, past_kv_list=kv_cache_vis, use_cache=True
        )
        attn_step = vis_model.blocks[0].attn.last_attn_weights[0]  # (heads, 1, P+step+1)
        decode_attn_steps.append(attn_step.cpu().numpy())
        curr_token = logits_d[:, -1, :].argmax(-1, keepdim=True)

# ── 可视化 ─────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))

# Prefill 注意力（Head 0）
ax_pf = fig.add_subplot(2, 3, 1)
attn_pf_np = attn_prefill[0].cpu().numpy()  # (P, P)
sns.heatmap(attn_pf_np, ax=ax_pf, cmap='Blues', vmin=0, vmax=1,
            xticklabels=[f'K{i}' for i in range(PROMPT_LEN_VIS)],
            yticklabels=[f'Q{i}' for i in range(PROMPT_LEN_VIS)],
            linewidths=0.5, linecolor='lightgray', cbar=True)
ax_pf.set_title(f'Prefill Phase\n(all {PROMPT_LEN_VIS} tokens, Head 0)', fontsize=11, fontweight='bold')
ax_pf.set_xlabel('Key Position')
ax_pf.set_ylabel('Query Position')

# Prefill 注意力（Head 1）
ax_pf2 = fig.add_subplot(2, 3, 4)
attn_pf2_np = attn_prefill[1].cpu().numpy()  # (P, P)
sns.heatmap(attn_pf2_np, ax=ax_pf2, cmap='Oranges', vmin=0, vmax=1,
            xticklabels=[f'K{i}' for i in range(PROMPT_LEN_VIS)],
            yticklabels=[f'Q{i}' for i in range(PROMPT_LEN_VIS)],
            linewidths=0.5, linecolor='lightgray', cbar=True)
ax_pf2.set_title(f'Prefill Phase\n(all {PROMPT_LEN_VIS} tokens, Head 1)', fontsize=11, fontweight='bold')
ax_pf2.set_xlabel('Key Position')
ax_pf2.set_ylabel('Query Position')

# Decode 步骤 1~4（Head 0）
for step_idx, (attn_step, ax_pos) in enumerate(zip(decode_attn_steps, [2, 3, 5, 6])):
    ax_d = fig.add_subplot(2, 3, ax_pos)
    total = PROMPT_LEN_VIS + step_idx + 1
    data = attn_step[0, 0, :total].reshape(1, -1)  # (1, total)
    sns.heatmap(data, ax=ax_d, cmap='Blues', vmin=0, vmax=1,
                xticklabels=[f'{i}' for i in range(total)],
                yticklabels=[f'Q{PROMPT_LEN_VIS + step_idx}'],
                linewidths=0.5, linecolor='lightgray', cbar=True)
    ax_d.set_title(
        f'Decode Step {step_idx+1}  (1 Query → {total} Keys, Head 0)',
        fontsize=10, fontweight='bold'
    )
    ax_d.set_xlabel('Key Position (cached + new)')
    # 标注 cached 区域
    ax_d.axvline(x=PROMPT_LEN_VIS + step_idx, color='red', linewidth=1.5, linestyle='--', alpha=0.7)
    ax_d.text(PROMPT_LEN_VIS + step_idx - 0.3, 0.5, '←cached', color='red',
              fontsize=8, va='center', ha='right')

plt.suptitle('Attention Patterns: Prefill vs Decode\n'
             '(Red dashed line = boundary between cached and new K/V)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('kv_cache_attn_patterns.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── KV Cache 增长可视化 ────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

prompt_size = 8
decode_steps = 10
total_steps = prompt_size + decode_steps

for step in range(total_steps):
    is_prompt = step < prompt_size
    color = '#4C72B0' if is_prompt else '#DD8452'
    label = 'Prompt (Prefill)' if step == 0 else ('Generated Token (Decode)' if step == prompt_size else '')
    ax.barh(0, 1, left=step, height=0.5, color=color, edgecolor='white', linewidth=1.5, label=label)
    ax.text(step + 0.5, 0, str(step), ha='center', va='center', fontsize=9, color='white', fontweight='bold')

# 标注 Cache 状态
for decode_step in range(decode_steps + 1):
    cache_len = prompt_size + decode_step
    y_pos = -(decode_step + 1) * 0.6
    for pos in range(cache_len):
        is_prompt = pos < prompt_size
        color = '#4C72B0' if is_prompt else '#DD8452'
        ax.barh(y_pos, 1, left=pos, height=0.45,
                color=color, alpha=0.7, edgecolor='white', linewidth=0.5)
    ax.text(-0.5, y_pos, f'After Step {decode_step}\n(cache={cache_len})',
            ha='right', va='center', fontsize=8, style='italic')
    ax.text(cache_len + 0.2, y_pos, f'↑ KV size = {cache_len}',
            ha='left', va='center', fontsize=8, color='gray')

ax.set_xlim(-5, total_steps + 3)
ax.set_ylim(-(decode_steps + 1) * 0.6 - 0.4, 0.6)
ax.set_yticks([])
ax.set_xlabel('Token Position', fontsize=12)
ax.set_title('KV Cache Growth Over Decode Steps\n'
             '(Blue = Prompt tokens, Orange = Generated tokens)', fontsize=12)
ax.legend(loc='upper right', fontsize=10)
ax.axvline(x=prompt_size, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.text(prompt_size, 0.55, 'Prompt\nEnd', ha='center', va='bottom', fontsize=9, color='gray')
ax.grid(axis='x', alpha=0.2)

plt.tight_layout()
plt.savefig('kv_cache_growth.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. 总结

### KV Cache 核心要点

| 方面 | 无 KV Cache | 有 KV Cache |
|------|------------|-------------|
| K/V 投影计算量 | $O(T^2)$（全部重算） | $O(T)$（只算新 token） |
| 注意力计算量 | $O(T^2)$（不可避免） | $O(T^2)$（不可避免） |
| 每步序列输入长度 | $T$（逐步增长） | $1$（始终为 1） |
| 额外内存开销 | 无 | $L \times 2 \times B \times d_{model} \times T$ |
| 实现复杂度 | 简单 | 需管理每层 Cache |

### 为什么 KV Cache 是正确的？

因果注意力保证了位置 $i$ 的隐状态只依赖于位置 $0, \ldots, i$ 的输入（与未来 token 无关），  
因此缓存的 $K_i, V_i$ 在后续步骤中**永远不需要重新计算**，可以安全复用。

### 现代 LLM 中的优化

- **Multi-Query Attention (MQA)**：所有头共享 K, V，大幅减少 KV Cache 内存
- **Grouped-Query Attention (GQA)**：分组共享 K, V（LLaMA-2/3 使用）
- **滑动窗口注意力**：只缓存最近 $w$ 个位置，限制 Cache 大小（Mistral 使用）
- **PagedAttention**：将 KV Cache 分页管理，提高 GPU 内存利用率（vLLM 使用）
- **Flash Attention**：通过分块计算减少显存占用，与 KV Cache 正交

### 延伸阅读

- [Efficient Transformers: A Survey](https://arxiv.org/abs/2009.06732)
- [GQA: Training Generalized Multi-Query Transformer Models](https://arxiv.org/abs/2305.13245)
- [vLLM: Efficient Memory Management for LLM Serving](https://arxiv.org/abs/2309.06180)
- [FlashAttention-2](https://arxiv.org/abs/2307.08691)